In [1]:
# Validator setup. Same isolation as notebook 03: the LLM only reads the
# catalog file, the notebook itself runs the SQL the model outputs.
import os
import re
import time
from pathlib import Path

import requests
import yaml
from dotenv import load_dotenv
import oracledb

# project root = parent of notebook/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

# opencode inference endpoint (OpenAI-compatible chat completions)
LLM_URL = os.getenv("LLM_URL")
LLM_KEY = os.getenv("SQL_AGENT")     # API key stored under the name SQL_AGENT
LLM_MODEL = os.getenv("LLM_MODEL", "qwen3.8-flash")

# fail fast if the LLM config is missing
missing = [n for n, v in {"LLM_URL": LLM_URL, "SQL_AGENT": LLM_KEY}.items() if not v]
if missing:
    raise ValueError(f"Missing variables in .env: {', '.join(missing)}")

CATALOG_PATH = PROJECT_ROOT / "catalog" / "catalog.yaml"
print(f"model: {LLM_MODEL} | catalog: {CATALOG_PATH.name}")

model: qwen3.8-flash | catalog: catalog.yaml


In [2]:
# The functions below are DEFINED in notebook/03_sql_writer.ipynb and
# exported to src/ by that notebook's export cell. Import them here instead
# of copying - src/ is the single source of truth for shared agent code.
import sys

sys.path.insert(0, str(PROJECT_ROOT))

from src.llm import llm
from src.rules import render_rules, pdp_class
from src.writer import schema_block, sql_rules, sql_writer

print("agent code imported from src/ (defined first in notebook 03)")

agent code imported from src/ (defined first in notebook 03)


In [3]:
# Load the catalog AND the PDP rules file (both live under catalog/)
catalog = yaml.safe_load(CATALOG_PATH.read_text(encoding="utf-8"))
RULES_PATH = PROJECT_ROOT / "catalog" / "rules.yaml"
rules = yaml.safe_load(RULES_PATH.read_text(encoding="utf-8"))

print("policy:", catalog["policy"])
print(f"rules: v{rules['rules_version']}")
for name, ws in catalog["workspaces"].items():
    print(f"{name}: {len(ws['tables'])} tables")

policy: {'max_workspaces_per_question': 2, 'cross_schema_joins': 'catalog_connections_only', 'cross_schema_fk_count': 0, 'connection_count': 3}
rules: v1.1
CO: 11 tables
HR: 8 tables
SH: 12 tables


In [4]:
# Safety guard and read-only execution. The validator never trusts the
# model's text: every attempt passes the same gate before touching the DB.
def is_safe_sql(sql):
    # must start with SELECT or WITH and contain no write/DDL keywords.
    # the pattern lives INSIDE the function so this cell exports verbatim to
    # src/validator.py (no module-level constants to carry along).
    if not re.match(r"\s*(SELECT|WITH)\b", sql, re.IGNORECASE):
        return False
    blocked = re.compile(
        r"\b(INSERT|UPDATE|DELETE|MERGE|DROP|ALTER|TRUNCATE|CREATE|GRANT|REVOKE|EXEC|EXECUTE)\b",
        re.IGNORECASE,
    )
    return not blocked.search(sql)

def execute_readonly(sql, workspace_names, catalog, max_rows=20):
    # a trailing semicolon is not allowed inside the wrapper subquery
    sql = sql.strip().rstrip(";").strip()
    if not is_safe_sql(sql):
        raise ValueError("blocked: not a SELECT/WITH statement (or contains write/DDL keywords)")
    names = [workspace_names] if isinstance(workspace_names, str) else list(workspace_names)
    wrapped = f"SELECT * FROM ({sql}) WHERE ROWNUM <= {max_rows}"
    with oracledb.connect(user=os.getenv("ORA_USER"),
                          password=os.getenv("ORA_PASSWORD"),
                          dsn=os.getenv("ORA_DSN")) as conn:
        with conn.cursor() as cur:
            # single-workspace queries use unqualified names: set the session
            # schema. Multi-workspace queries are SCHEMA.TABLE qualified.
            if len(names) == 1:
                ws = names[0]
                if not re.fullmatch(r"[A-Za-z0-9_]+", ws) or ws not in catalog["workspaces"]:
                    raise ValueError(f"blocked: invalid workspace {ws!r}")
                cur.execute(f"ALTER SESSION SET CURRENT_SCHEMA = {ws}")
            cur.execute(wrapped)
            columns = [d[0] for d in cur.description]
            rows = cur.fetchall()
    return columns, rows

In [5]:
# The validator agent: run, check, and repair (max N retries).\n# rules is a parameter, not a global: this cell exports verbatim to src/validator.py.
# Two failure modes are repaired differently:
# - error:     SQL text fed back with the database error message
# - empty:     SQL ran but 0 rows -> ask the model to reconsider filters,
#              joins and date logic (real case: TIMES covers years with no SALES)
def repair_sql(question, sql, problem, workspace_names, catalog, rules):
    schema = schema_block(workspace_names, catalog, rules)
    rules_text = render_rules(rules)
    prompt = (
        "You fix SQL queries.\n\n"
        f"{rules_text}\n\n"
        f"{sql_rules(catalog['policy'])}\n\n"
        f"Question: {question}\n\n"
        f"Schema:\n{schema}\n\n"
        f"Previous SQL:\n{sql}\n\n"
        f"Problem:\n{problem}\n\n"
        "Return ONLY the corrected SQL query."
    )
    answer = llm(prompt, system="You fix SQL queries.", session="ses_queryllm_validator")
    answer = re.sub(r"^```(?:sql)?\s*|\s*```$", "", answer.strip(), flags=re.IGNORECASE)
    return answer.strip().rstrip(";").strip()

def validator_agent(question, sql, workspace_names, catalog, rules, max_retries=3):
    history = []
    for attempt in range(1, max_retries + 1):
        try:
            columns, rows = execute_readonly(sql, workspace_names, catalog)
            if rows:
                return {
                    "status": "ok",
                    "sql": sql,
                    "columns": columns,
                    "rows": rows,
                    "attempts": attempt,
                    "history": history,
                }
            problem = ("The query executed but returned 0 rows. Reconsider the "
                       "filters, joins and date logic so the question is answered "
                       "with the data that exists.")
            history.append({"attempt": attempt, "sql": sql, "outcome": "empty"})
        except Exception as exc:
            problem = str(exc).strip() or type(exc).__name__
            history.append({"attempt": attempt, "sql": sql, "outcome": f"error: {problem[:160]}"})
        if attempt == max_retries:
            break
        print(f"  attempt {attempt} -> {history[-1]['outcome'][:60]} | asking the LLM to repair")
        sql = repair_sql(question, sql, problem, workspace_names, catalog, rules)
    return {"status": "failed", "sql": sql, "history": history}

In [6]:
# End-to-end demo: writer -> validator. The SH question is deliberately
# hard: TIMES covers calendar years the SALES table has no data for, so the
# first SQL usually runs but returns 0 rows - the empty-result repair case.
QUESTIONS = [
    ("CO", "Total revenue per month for each store"),
    ("HR", "Number of employees and average salary per department"),
    ("SH", "Top 5 products by revenue in the last calendar year"),
]

results = {}
for ws_name, question in QUESTIONS:
    print(f"[{ws_name}] {question}")
    draft = sql_writer(question, ws_name, catalog, rules)
    outcome = validator_agent(question, draft, ws_name, catalog, rules, max_retries=3)
    results[question] = outcome
    tries = outcome.get("attempts", len(outcome["history"]))
    print(f"validator: {outcome['status']} after {tries} attempt(s)")
    print("final SQL:")
    print(outcome["sql"])
    if outcome["status"] == "ok":
        print("preview rows:")
        for row in outcome["rows"][:5]:
            print("  ", row)
    print("-" * 60)

[CO] Total revenue per month for each store


validator: ok after 1 attempt(s)
final SQL:
SELECT s.store_name,
       TO_CHAR(o.order_tms, 'YYYY-MM') AS order_month,
       SUM(oi.unit_price * oi.quantity) AS total_revenue
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN stores s ON s.store_id = o.store_id
GROUP BY s.store_name, TO_CHAR(o.order_tms, 'YYYY-MM')
ORDER BY order_month, s.store_name
preview rows:
   ('Online', '2021-02', 3256.22)
   ('Online', '2021-03', 11154.24)
   ('Seattle', '2021-03', 251.18)
   ('New York City', '2021-04', 33.56)
   ('Online', '2021-04', 18283.83)
------------------------------------------------------------
[HR] Number of employees and average salary per department


validator: ok after 1 attempt(s)
final SQL:
SELECT d.DEPARTMENT_NAME,
       COUNT(e.EMPLOYEE_ID) AS EMPLOYEE_COUNT,
       AVG(e.SALARY) AS AVG_SALARY
FROM EMPLOYEES e
JOIN DEPARTMENTS d ON d.DEPARTMENT_ID = e.DEPARTMENT_ID
GROUP BY d.DEPARTMENT_ID, d.DEPARTMENT_NAME
HAVING COUNT(*) >= 5
ORDER BY d.DEPARTMENT_NAME
preview rows:
   ('Finance', 6, 8601.333333333334)
   ('IT', 5, 5760)
   ('Purchasing', 6, 4150)
   ('Sales', 34, 8955.882352941177)
   ('Shipping', 45, 3475.5555555555557)
------------------------------------------------------------
[SH] Top 5 products by revenue in the last calendar year


  attempt 1 -> empty | asking the LLM to repair


  llm call failed (attempt 1/3), retrying...


validator: ok after 2 attempt(s)
final SQL:
WITH last_sales_year AS (
    SELECT MAX(t.CALENDAR_YEAR) AS LY
    FROM SALES s
    JOIN TIMES t ON s.TIME_ID = t.TIME_ID
)
SELECT p.PROD_NAME,
       SUM(s.AMOUNT_SOLD) AS REVENUE
FROM SALES s
JOIN PRODUCTS p ON s.PROD_ID = p.PROD_ID
JOIN TIMES t ON s.TIME_ID = t.TIME_ID
WHERE t.CALENDAR_YEAR = (SELECT LY FROM last_sales_year)
GROUP BY p.PROD_ID, p.PROD_NAME
ORDER BY REVENUE DESC
FETCH FIRST 5 ROWS ONLY
preview rows:
   ('Lithium Electric Golf Caddy', 3453656.62)
   ('Soccer Goal - Official', 2819074.98)
   ('Right-Handed Graphite Shaft Iron Set', 2230713.39)
   ('Soccer Goal - Portable', 2205836.66)
   ('Pitching Machine and Batting Cage Combo', 1874621.96)
------------------------------------------------------------


In [7]:
# Summary: how many attempts each question needed and what failed
total_attempts = 0
for question, outcome in results.items():
    status = outcome["status"]
    tries = outcome.get("attempts", len(outcome["history"]))
    total_attempts += tries
    print(f"[{status}] attempts={tries} :: {question}")
    for h in outcome["history"]:
        print(f"    attempt {h['attempt']}: {h['outcome']}")
print(f"\n{sum(1 for o in results.values() if o['status'] == 'ok')}/{len(results)} questions validated")

[ok] attempts=1 :: Total revenue per month for each store
[ok] attempts=1 :: Number of employees and average salary per department
[ok] attempts=2 :: Top 5 products by revenue in the last calendar year
    attempt 1: empty

3/3 questions validated


In [8]:
# Export to src/: the functions defined above in this notebook (first
# definitions) are written to src/validator.py so later notebooks import
# them instead of copying. Re-run this notebook after changing them.
import inspect

SRC_DIR = PROJECT_ROOT / "src"
def export_module(filename, header, names):
    parts = [header]
    for name in names:
        parts.append("\n\n" + inspect.getsource(globals()[name]).rstrip() + "\n")
    (SRC_DIR / filename).write_text("".join(parts), encoding="utf-8")
    print(f"exported src/{filename}: {', '.join(names)}")

export_module(
    "validator.py",
    "# Validator agent (first defined in notebook/04_validator.ipynb).\n"
    "# is_safe_sql + execute_readonly are the deterministic guard every SQL\n"
    "# attempt must pass; repair_sql + validator_agent are the repair loop.\n"
    "import os\nimport re\n\nimport oracledb\n\n"
    "from src.llm import llm\n"
    "from src.rules import render_rules\n"
    "from src.writer import schema_block, sql_rules\n",
    ["is_safe_sql", "execute_readonly", "repair_sql", "validator_agent"],
)

exported src/validator.py: is_safe_sql, execute_readonly, repair_sql, validator_agent
